# MPEC vs. Nested Fixed Point Simulation Tutorial

In [1]:
import pyblp
import numpy as np
import pandas as pd

pyblp.options.digits = 2
pyblp.options.verbose = False
pyblp.__version__

'1.2.0'

The [Problem Simulation Tutorial](simulation.ipynb) shows how to use :class:`Simulation` to verify that true parameters can be recovered from simulated data, and the [MPEC vs. Nested Fixed Point Tutorial](mpec.ipynb) shows how :meth:`Problem.solve` can estimate $\hat{\theta}$ with either the default nested fixed point (NFP) approach or the MPEC (mathematical program with equilibrium constraints) approach of :ref:`references:Dubé, Fox, and Su (2012)`. That tutorial compares the two approaches on real data, where the true parameters are unknown, so it can only check that NFP and MPEC *agree with each other*.

In this tutorial, we'll combine both ideas: we'll simulate an economy with known true parameters, and use it to check that NFP and MPEC not only agree with each other, but also both recover the truth.

As described in :ref:`background:MPEC`, the MPEC implementation in pyblp currently only supports demand-side estimation with $\hat{\beta}$ fully concentrated out: no supply side, no micro moments, and no covariance moments. We'll simulate a full supply and demand system so that equilibrium prices are realistic, but we'll only estimate the demand side.

## Simulating an Economy

As in the Problem Simulation Tutorial, we'll use :func:`build_id_data` to build market and firm IDs for a model with $T = 50$ markets, each with $J_t = 20$ products produced by $F = 10$ firms.

In [2]:
id_data = pyblp.build_id_data(T=50, J=20, F=10)

Next, we'll create an :class:`Integration` configuration to build agent data according to a Gauss-Hermite product rule.

In [3]:
integration = pyblp.Integration('product', 9)
integration

Configured to construct nodes and weights according to the level-9 Gauss-Hermite product rule with options {}.

We'll configure the same demand and supply specification as the Problem Simulation Tutorial: an $X_1$ of a constant, prices, and an exogenous characteristic; an $X_2$ with a random coefficient on that same characteristic; and an $X_3$ with the characteristic and a cost-shifter. The supply side is only used to simulate realistic equilibrium prices; we'll drop it when we build the :class:`Problem` below.

In [4]:
X1_formulation = pyblp.Formulation('1 + prices + x')
X2_formulation = pyblp.Formulation('0 + x')
X3_formulation = pyblp.Formulation('0 + x + z')

simulation = pyblp.Simulation(
    product_formulations=(X1_formulation, X2_formulation, X3_formulation),
    beta=[1, -2, 2],
    sigma=1,
    gamma=[1, 4],
    product_data=id_data,
    integration=integration,
    seed=0,
)
simulation

Dimensions:
 T    N     F    I    K1    K2    K3 
---  ----  ---  ---  ----  ----  ----
50   1000  10   450   3     1     2  

Formulations:
        Column Indices:           0     1      2 
-------------------------------  ---  ------  ---
  X1: Linear Characteristics      1   prices   x 
 X2: Nonlinear Characteristics    x              
X3: Linear Cost Characteristics   x     z        

Nonlinear Coefficient True Values:
Sigma:     x    
------  --------
  x     +1.0E+00

Beta True Values:
   1       prices      x    
--------  --------  --------
+1.0E+00  -2.0E+00  +2.0E+00

Gamma True Values:
   x         z    
--------  --------
+1.0E+00  +4.0E+00

Simulated prices and shares aren't yet consistent with the true parameters, so we solve for equilibrium with :meth:`Simulation.replace_endogenous`, which iterates over the $\zeta$-markup equation from :ref:`references:Morrow and Skerlos (2011)`.

In [5]:
simulation_results = simulation.replace_endogenous()
simulation_results

Simulation Results Summary:
Computation  Fixed Point  Fixed Point  Contraction  Profit Gradients  Profit Hessians  Profit Hessians
   Time       Failures    Iterations   Evaluations      Max Norm      Min Eigenvalue   Max Eigenvalue 
-----------  -----------  -----------  -----------  ----------------  ---------------  ---------------
 00:00:00         0           721          721          +1.3E-13         -8.4E-01         -9.6E-06    

## Building a Demand-Only Problem

Because MPEC currently only supports demand-side estimation, we'll pass just the demand-side formulations to :meth:`SimulationResults.to_problem`, which by default would otherwise reuse all three of :attr:`Simulation.product_formulations`. This gives us a :class:`Problem` with no supply side ($K_3 = 0$), even though the underlying simulated prices came from a full supply and demand equilibrium.

In [6]:
problem = simulation_results.to_problem(product_formulations=(X1_formulation, X2_formulation))
problem

Dimensions:
 T    N     F    I    K1    K2    MD 
---  ----  ---  ---  ----  ----  ----
50   1000  10   450   3     1     5  

Formulations:
       Column Indices:          0     1      2 
-----------------------------  ---  ------  ---
 X1: Linear Characteristics     1   prices   x 
X2: Nonlinear Characteristics   x              

## Solving with the Nested Fixed Point Approach

We'll start $\Sigma_0$ at half its true value so that the optimization routine has to do some work. Because the problem has no supply side, $\hat{\beta}$, including the coefficient on prices, is fully concentrated out, so we don't need to specify any starting values for it.

In [7]:
bfgs = pyblp.Optimization('bfgs', {'gtol': 1e-4})
nfp_results = problem.solve(sigma=0.5 * simulation.sigma, optimization=bfgs)
nfp_results

Problem Results Summary:
GMM   Objective  Gradient            Clipped  Weighting Matrix  Covariance Matrix
Step    Value      Norm    Hessian   Shares   Condition Number  Condition Number 
----  ---------  --------  --------  -------  ----------------  -----------------
 2    +4.0E+00   +1.6E-07  +7.6E+00     0         +8.4E+03          +3.2E+03     

Cumulative Statistics:
Computation  Optimizer  Optimization   Objective   Fixed Point  Contraction
   Time      Converged   Iterations   Evaluations  Iterations   Evaluations
-----------  ---------  ------------  -----------  -----------  -----------
 00:00:00       Yes          8            15          4268         13485   

Nonlinear Coefficient Estimates (Robust SEs in Parentheses):
Sigma:      x     
------  ----------
  x      +1.1E+00 
        (+5.1E-01)

Beta Estimates (Robust SEs in Parentheses):
    1         prices        x     
----------  ----------  ----------
 +1.0E+00    -2.0E+00    +2.0E+00 
(+1.3E-01)  (+2.9E-02)  (+1.6E-

## Solving with MPEC

We configure MPEC with ``Optimization('mpec-trust-constr')`` and start from the same $\Sigma_0$ and the default starting $\delta$ (the closed-form plain logit solution) as the NFP problem above.

In [8]:
mpec_cold_results = problem.solve(
    sigma=0.5 * simulation.sigma,
    optimization=pyblp.Optimization('mpec-trust-constr'),
)
mpec_cold_results

Problem Results Summary:
GMM   Objective    Projected    Reduced   Clipped  Weighting Matrix  Covariance Matrix
Step    Value    Gradient Norm  Hessian   Shares   Condition Number  Condition Number 
----  ---------  -------------  --------  -------  ----------------  -----------------
 2    +4.0E+00     +4.5E-08     +7.6E+00     0         +8.4E+03          +3.2E+03     

Cumulative Statistics:
Computation  Optimizer  Optimization   Objective   Fixed Point  Contraction
   Time      Converged   Iterations   Evaluations  Iterations   Evaluations
-----------  ---------  ------------  -----------  -----------  -----------
 00:00:00       Yes          42           26            0           100    

Nonlinear Coefficient Estimates (Robust SEs in Parentheses):
Sigma:      x     
------  ----------
  x      +1.1E+00 
        (+5.1E-01)

Beta Estimates (Robust SEs in Parentheses):
    1         prices        x     
----------  ----------  ----------
 +1.0E+00    -2.0E+00    +2.0E+00 
(+1.3E-01) 

Unlike the higher-dimensional real-data example in the MPEC vs. Nested Fixed Point Tutorial, cold-started MPEC here converges to essentially the same point as NFP: with only one nonlinear parameter, this simulated problem is much better behaved. As that tutorial notes, MPEC's optimizer can still be sensitive to starting values on harder problems, and [Artleys Knitro](https://www.artelys.com/solvers/knitro/) (``Optimization('mpec-knitro')``) tends to be more robust than SciPy's ``trust-constr`` used here. To directly verify the equivalence result of :ref:`references:Su and Judd (2012)`, we'll also warm-start MPEC from the NFP estimates.

In [9]:
mpec_warm_results = problem.solve(
    sigma=nfp_results.sigma,
    optimization=pyblp.Optimization('mpec-trust-constr'),
    delta=nfp_results.delta,
)
mpec_warm_results

Problem Results Summary:
GMM   Objective    Projected    Reduced   Clipped  Weighting Matrix  Covariance Matrix
Step    Value    Gradient Norm  Hessian   Shares   Condition Number  Condition Number 
----  ---------  -------------  --------  -------  ----------------  -----------------
 2    +4.0E+00     +4.5E-08     +7.6E+00     0         +8.4E+03          +3.2E+03     

Cumulative Statistics:
Computation  Optimizer  Optimization   Objective   Fixed Point  Contraction
   Time      Converged   Iterations   Evaluations  Iterations   Evaluations
-----------  ---------  ------------  -----------  -----------  -----------
 00:00:00       Yes          34           18            0           100    

Nonlinear Coefficient Estimates (Robust SEs in Parentheses):
Sigma:      x     
------  ----------
  x      +1.1E+00 
        (+5.1E-01)

Beta Estimates (Robust SEs in Parentheses):
    1         prices        x     
----------  ----------  ----------
 +1.0E+00    -2.0E+00    +2.0E+00 
(+1.3E-01) 

## Comparing Estimates

Because we simulated this economy ourselves, we know the true parameters, so we can compare NFP and MPEC not only against each other but also against the truth.

In [10]:
comparison = pd.DataFrame({
    'NFP': [
        float(nfp_results.objective),
        float(nfp_results.sigma[0, 0]),
        float(nfp_results.beta[0, 0]),
        float(nfp_results.beta[1, 0]),
        float(nfp_results.beta[2, 0]),
    ],
    'MPEC (cold start)': [
        float(mpec_cold_results.objective),
        float(mpec_cold_results.sigma[0, 0]),
        float(mpec_cold_results.beta[0, 0]),
        float(mpec_cold_results.beta[1, 0]),
        float(mpec_cold_results.beta[2, 0]),
    ],
    'MPEC (warm start)': [
        float(mpec_warm_results.objective),
        float(mpec_warm_results.sigma[0, 0]),
        float(mpec_warm_results.beta[0, 0]),
        float(mpec_warm_results.beta[1, 0]),
        float(mpec_warm_results.beta[2, 0]),
    ],
    'Truth': [
        np.nan,
        float(simulation.sigma[0, 0]),
        float(simulation.beta[0, 0]),
        float(simulation.beta[1, 0]),
        float(simulation.beta[2, 0]),
    ],
}, index=['objective', 'sigma[0, 0]', 'beta (constant)', 'beta (prices)', 'beta (x)'])
comparison

,NFP,MPEC (cold start),MPEC (warm start),Truth
objective,4.032187,4.032186,4.032186,NaN
"sigma[0, 0]",1.131895,1.131895,1.131895,1.0
beta (constant),1.044169,1.044169,1.044169,1.0
beta (prices),-2.015312,-2.015312,-2.015312,-2.0
beta (x),2.006918,2.006918,2.006918,2.0


All three sets of estimates are close to the truth, and warm-started MPEC agrees with NFP to several decimal places, consistent with :ref:`references:Su and Judd (2012)`'s equivalence result. We can also check the maximum absolute differences directly, including for $\xi$ and standard errors.

In [11]:
print('max |sigma difference| =', np.nanmax(np.abs(nfp_results.sigma - mpec_warm_results.sigma)))
print('max |sigma_se difference| =', np.nanmax(np.abs(nfp_results.sigma_se - mpec_warm_results.sigma_se)))
print('max |beta difference| =', np.nanmax(np.abs(nfp_results.beta - mpec_warm_results.beta)))
print('max |beta_se difference| =', np.nanmax(np.abs(nfp_results.beta_se - mpec_warm_results.beta_se)))
print('max |xi difference| =', np.nanmax(np.abs(nfp_results.xi - mpec_warm_results.xi)))

max |sigma difference| = 1.3342207783040294e-08
max |sigma_se difference| = 1.1821404966028126e-09
max |beta difference| = 2.044350289054364e-08
max |beta_se difference| = 9.132787304189094e-10
max |xi difference| = 2.8433765564273017e-08


The remaining differences between NFP and warm-started MPEC are on the order of the two solvers' default optimization tolerances, not a meaningful economic difference, and both recover parameters close to the truth we simulated. As in the MPEC vs. Nested Fixed Point Tutorial, the choice between NFP and MPEC is a purely computational one: both target the same GMM estimator, and, as with any nonconvex estimation problem, it is a good idea to try multiple starting values with either approach.